# 03 Explainable AI

This notebook provides global and local explanation patterns. Because features are anonymized, explanations are governance signals that must be mapped back to real stations before deployment.

In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
from src.config import SAMPLE_DIR
from src.preprocessing import (
    sample_training_data,
    feature_engineer,
    split_features_target,
    prepare_model_matrix,
)
from src.explainability import permutation_importance_table, shap_summary

In [ ]:
try:
    df = pd.read_csv(SAMPLE_DIR / "bosch_training_sample.csv")
except FileNotFoundError:
    df = sample_training_data(n_rows=20000)

df = feature_engineer(df)
X, y = split_features_target(df)
X_model = prepare_model_matrix(X)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X_train, X_valid, y_train, y_valid = train_test_split(
    X_model, y, test_size=0.25, random_state=42, stratify=y
)
model = RandomForestClassifier(
    n_estimators=120, max_depth=8, class_weight="balanced_subsample", random_state=42, n_jobs=-1
)
model.fit(X_train, y_train)

## Global Explainability

In [ ]:
importance = permutation_importance_table(model, X_valid.head(1000), y_valid.head(1000), top_n=20)
importance

## Local Explanation For High-Risk Components

In [ ]:
scores = model.predict_proba(X_valid)[:, 1]
high_risk_index = scores.argmax()
local_row = X_valid.iloc[[high_risk_index]]
local_score = scores[high_risk_index]
local_score, local_row.T.sort_values(local_row.index[0], ascending=False).head(15)

In [ ]:
shap_values = shap_summary(model, X_valid.head(200))
(
    type(shap_values),
    shap_values
    if isinstance(shap_values, dict)
    else "SHAP values computed; use shap.plots in an interactive environment.",
)

## Governance Recommendation

Any high-impact anonymized feature must be mapped back to a real line, station, test, sensor, and process owner. Without that mapping, the model can prioritize inspection but cannot provide a responsible root-cause recommendation.